# Pertemuan 12 – Asosiasi Data dan Sistem Rekomendasi Dasar

**Nama:** Amin Siddik Rangkuti  
**NIM:** 220401010124  
**Program Studi:** PJJ Informatika  
**Mata Kuliah:** Pengantar Data Science

Notebook ini membahas Market Basket Analysis menggunakan algoritma Apriori, pembentukan association rules, serta sistem rekomendasi sederhana berbasis Content-Based Filtering.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aminsiddik2810/data-science-2026/blob/main/Pertemuan12_AminSiddikRangkuti_220401010124.ipynb)

## Langkah 0 – Instalasi Library
Library `mlxtend` digunakan untuk menjalankan algoritma Apriori dan membuat aturan asosiasi.

In [ ]:
!pip install mlxtend -q

## Langkah 1 – Generate dan Eksplorasi Dataset Transaksi
Dataset transaksi dibuat secara sintetis menggunakan pola pembelian tersembunyi. Produk **Roti** dibuat cenderung muncul bersama **Selai** agar terbentuk pola asosiasi yang dapat ditemukan oleh algoritma Apriori.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
          'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, setiap transaksi berisi 2–5 produk
transaksi = []
for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(list(np.random.choice(produk, n_item, replace=False)))

# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print('Contoh transaksi:')
for i, trx in enumerate(transaksi[:5], start=1):
    print(f'Transaksi {i}: {trx}')

print('\nJumlah transaksi:', len(transaksi))

### Eksplorasi Frekuensi Produk
Frekuensi setiap produk dihitung untuk melihat item yang paling sering muncul dalam transaksi.

In [ ]:
frekuensi_produk = pd.Series(
    [item for trx in transaksi for item in trx]
).value_counts()

display(frekuensi_produk.to_frame('Frekuensi'))

plt.figure(figsize=(10, 5))
frekuensi_produk.sort_values().plot(kind='barh')
plt.title('Frekuensi Kemunculan Produk')
plt.xlabel('Jumlah Kemunculan')
plt.ylabel('Produk')
plt.tight_layout()
plt.show()

## Langkah 2 – One-Hot Encoding Transaksi
Daftar transaksi diubah menjadi tabel one-hot encoding menggunakan `TransactionEncoder`. Nilai `True` berarti produk terdapat dalam transaksi, sedangkan `False` berarti produk tidak terdapat dalam transaksi.

In [ ]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

print('Ukuran data one-hot:', df.shape)
display(df.head(10))

## Langkah 3 – Mencari Frequent Itemset dengan Apriori
Algoritma Apriori dijalankan menggunakan beberapa nilai `min_support` untuk melihat pengaruh ambang batas support terhadap jumlah frequent itemset yang terbentuk.

In [ ]:
from mlxtend.frequent_patterns import apriori

hasil_support = []

for ms in [0.05, 0.10, 0.20]:
    freq = apriori(df, min_support=ms, use_colnames=True)
    hasil_support.append({
        'min_support': ms,
        'jumlah_itemset': len(freq)
    })
    print(f'min_support={ms:.2f} -> {len(freq)} itemset ditemukan')

pd.DataFrame(hasil_support)

In [ ]:
# Gunakan min_support yang menghasilkan jumlah itemset wajar
freq_items = apriori(df, min_support=0.10, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)

display(freq_items.head(15))

### Analisis Min Support
- Nilai `min_support` yang kecil menghasilkan lebih banyak itemset, tetapi sebagian dapat berupa pola yang lemah atau noise.
- Nilai `min_support` yang besar menghasilkan lebih sedikit itemset dan dapat melewatkan pola penting.
- Pada praktikum ini digunakan `min_support = 0.10` agar jumlah itemset tetap cukup banyak tetapi masih mudah dianalisis.

## Langkah 4 – Membentuk dan Menyaring Aturan Asosiasi
Aturan asosiasi dibentuk dari frequent itemset, kemudian disaring menggunakan `confidence` dan `lift`. Aturan dengan `lift > 1` menunjukkan hubungan positif antara antecedent dan consequent.

In [ ]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_items,
    metric='confidence',
    min_threshold=0.5,
    num_itemsets=len(df)
)

rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

kolom_tampil = ['antecedents', 'consequents', 'support', 'confidence', 'lift']
display(rules[kolom_tampil].head(15))

### Interpretasi Aturan Terkuat
Aturan dengan nilai `lift` tertinggi merupakan aturan yang hubungan antara antecedent dan consequent-nya paling kuat dibandingkan jika kedua item muncul secara independen.

In [ ]:
if not rules.empty:
    aturan_terkuat = rules.iloc[0]
    print('Aturan terkuat:')
    print(f"Antecedent : {set(aturan_terkuat['antecedents'])}")
    print(f"Consequent : {set(aturan_terkuat['consequents'])}")
    print(f"Support    : {aturan_terkuat['support']:.3f}")
    print(f"Confidence : {aturan_terkuat['confidence']:.3f}")
    print(f"Lift       : {aturan_terkuat['lift']:.3f}")
else:
    print('Tidak ada aturan yang memenuhi confidence >= 0.5 dan lift > 1.')

## Langkah 5 – Rekomendasi Sederhana dengan Content-Based Filtering
Sistem rekomendasi Content-Based Filtering merekomendasikan produk berdasarkan kemiripan kategori produk. Data kategori diubah menjadi one-hot encoding, lalu kemiripan dihitung menggunakan cosine similarity.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
               'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'],
    'kategori': ['Bakery', 'Bakery', 'Dairy', 'Cereal', 'Protein',
                 'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

display(katalog)
display(pd.DataFrame(sim_matrix, index=katalog['produk'], columns=katalog['produk']))

In [ ]:
def rekomendasi_serupa(nama_produk, top_n=3):
    if nama_produk not in katalog['produk'].values:
        return f'Produk {nama_produk} tidak ditemukan.'

    idx = katalog.index[katalog['produk'] == nama_produk][0]
    skor = list(enumerate(sim_matrix[idx]))
    skor = sorted(skor, key=lambda x: x[1], reverse=True)
    skor = [s for s in skor if s[0] != idx][:top_n]

    hasil = katalog.iloc[[i for i, _ in skor]][['produk', 'kategori']].copy()
    hasil['similarity'] = [nilai for _, nilai in skor]
    return hasil

print("Rekomendasi produk yang mirip dengan 'Roti':")
display(rekomendasi_serupa('Roti'))

## Langkah 6 – Membandingkan Kedua Pendekatan
Pada langkah ini, rekomendasi dari aturan asosiasi dibandingkan dengan hasil Content-Based Filtering untuk produk target **Roti**.

In [ ]:
produk_target = 'Roti'

# Rekomendasi dari association rules
rules_target = rules[
    rules['antecedents'].apply(lambda x: produk_target in x)
]

rekomendasi_rules = []
for itemset in rules_target['consequents']:
    rekomendasi_rules.extend(list(itemset))

rekomendasi_rules = list(dict.fromkeys(rekomendasi_rules))

print('Rekomendasi dari Association Rules:')
print(rekomendasi_rules[:5] if rekomendasi_rules else 'Tidak ada rekomendasi')

print('\nRekomendasi dari Content-Based Filtering:')
display(rekomendasi_serupa(produk_target))

### Analisis Perbandingan

**Association Rules** menghasilkan rekomendasi berdasarkan pola produk yang benar-benar sering dibeli bersama dalam transaksi. Hasilnya dipengaruhi oleh nilai support, confidence, dan lift.

**Content-Based Filtering** menghasilkan rekomendasi berdasarkan kesamaan kategori atau atribut produk. Pendekatan ini tetap dapat bekerja walaupun belum tersedia banyak transaksi pengguna.

Hasil kedua metode tidak selalu konsisten karena sumber informasinya berbeda. Association Rules menggunakan pola pembelian, sedangkan Content-Based Filtering menggunakan atribut produk. Keduanya dapat digabungkan menjadi sistem **hybrid** agar rekomendasi lebih lengkap dan relevan.

## Kesimpulan

1. Algoritma Apriori dapat digunakan untuk menemukan frequent itemset dari data transaksi tanpa label.
2. Support menunjukkan seberapa sering itemset muncul, confidence menunjukkan kemungkinan consequent muncul setelah antecedent, sedangkan lift mengukur kekuatan hubungan dibandingkan kondisi independen.
3. Nilai `min_support` yang terlalu kecil menghasilkan banyak pola dan noise, sedangkan nilai yang terlalu besar dapat menghilangkan pola penting.
4. Association Rules cocok untuk rekomendasi berdasarkan perilaku transaksi pelanggan.
5. Content-Based Filtering cocok untuk merekomendasikan item yang mempunyai atribut atau kategori serupa.
6. Pendekatan hybrid dapat menggabungkan kelebihan kedua metode untuk menghasilkan rekomendasi yang lebih baik.